# 4-3 マスタ結合と集計

4-2 で **47 ファイルが束ねられた `all_sales`** ができました。ただ、これだけでは:

- **県コードしかなくて県名が分からない** → 「13 都」と言われても通じない
- **商品コードしかなくてカテゴリが分からない** → 「定番果物 vs 高級果物」の比較ができない
- **原価情報が無いので利益が計算できない**

そこで **マスタ結合 (`merge`)** の出番です。Excel で言う **VLOOKUP / XLOOKUP** に相当する操作。pandas なら **1 行** で済みます。

## Excel との対比

| やりたいこと | Excel | pandas |
|---|---|---|
| マスタから値を引く | `VLOOKUP` / `XLOOKUP` | `df.merge(master, on="キー")` |
| 複数キーで集計 | ピボットテーブル | `groupby(["列A","列B"])` |
| 複数の指標を一気に出す | ピボットの値フィールド複数 | `.agg(売上=...,  利益=..., 件数=...)` |
| 縦持ち → 横持ち（クロス集計） | ピボットテーブル | `pivot_table(...)` |

ピボットテーブルが好きな人は、**`pivot_table` を覚えるだけで Excel の半分は再現できる** ようになります。

## 準備 — 4-2 までの読み込みを 1 セルに

In [ ]:
# === Colab ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル ===
BASE = "."

DATA = f"{BASE}/data/capstone"
SALES_DIR = f"{DATA}/sales_2026-01"

import pandas as pd
from pathlib import Path

# 4-2 でやった処理を 1 つにまとめたもの
files = sorted(Path(SALES_DIR).glob("sales_2026-01_*.xlsx"))
dfs = []
for f in files:
    df = pd.read_excel(f, sheet_name="売上")
    df["prefecture_code"] = int(f.stem.split("_")[2])
    dfs.append(df)
all_sales = pd.concat(dfs, ignore_index=True)
print(f"all_sales: {len(all_sales):,} 行")

In [ ]:
# マスタの読み込み
master_products = pd.read_excel(f"{DATA}/master_products.xlsx")
master_branches = pd.read_excel(f"{DATA}/master_branches.xlsx")

print("商品マスタ:")
print(master_products)
print()
print("支店マスタ (先頭5行):")
print(master_branches.head())

## 1. 商品マスタを結合する — `merge`

`all_sales.merge(master_products, on="product_code")` と書くと、**`product_code` をキーにして** マスタの全列を `all_sales` 側に持ってきます。

Excel の VLOOKUP で言うと:

```
=VLOOKUP(B2, 商品マスタ!A:E, 3, FALSE)  ← カテゴリ列を引く
=VLOOKUP(B2, 商品マスタ!A:E, 5, FALSE)  ← 原価列を引く
```

を **全行・全列まとめて 1 行で書く** イメージです。

In [ ]:
# 商品マスタから category と cost だけ持ってくる
df = all_sales.merge(
    master_products[["product_code", "category", "cost"]],
    on="product_code",
    how="left",   # 左(売上)の全行を必ず残す = VLOOKUP と同じ挙動
)
df.head()

**ポイント**:

- `master_products[["product_code", "category", "cost"]]` で **必要な列だけ抜く**。重複する列 (`product_name` `unit_price`) は売上側に既にあるので除外
- `on="product_code"` で **共通キーを指定**
- `how="left"` は **左 (= `all_sales`) の行を全部残す**。Excel の VLOOKUP と同じ感覚。これを `"inner"` にすると、マスタにない商品コードの行は消える

## 2. 支店マスタも結合する

同じ要領で **支店マスタ** を結合します。キーは `prefecture_code`。

In [ ]:
df = df.merge(master_branches, on="prefecture_code", how="left")
df.head()

**県名** (`prefecture_name`) と **地域ブロック** (`region`) が末尾に追加されました。これで `df` は **「いつ・どこで・誰が・何を・いくらで・どれだけ売って・カテゴリは何で・原価はいくらか」が全部入った 1 枚の表** になりました。

## 3. 利益列を追加する

原価が手に入ったので、**利益 = 売上 - 原価×数量** を計算して列にできます (2-4 の復習)。

In [ ]:
df["profit"] = df["amount"] - df["cost"] * df["quantity"]
df[["date", "product_name", "quantity", "amount", "cost", "profit"]].head()

## 4. `groupby + agg` — 複数の指標を一気に集計

ここからが分析の本番。`groupby` で **地域別の売上・利益・件数を一気に** 出してみます。

**コツ**: `.agg(集計後の列名=("元の列", "集計方法"))` という書き方を使うと、**Excel ピボットの「値フィールド」を複数並べた状態** を 1 行で書けます。

In [ ]:
by_region = df.groupby("region").agg(
    売上=("amount", "sum"),
    利益=("profit", "sum"),
    件数=("amount", "count"),
).sort_values("売上", ascending=False)

by_region

**1 行で「地域別 売上・利益・件数 ランキング」** が出ました。Excel でピボットテーブルを作って値フィールドを 3 つ並べる作業に相当します。

In [ ]:
# 県別 売上 TOP10 (県名で見たいので prefecture_name でグループ化)
by_pref = df.groupby("prefecture_name").agg(
    売上=("amount", "sum"),
    利益=("profit", "sum"),
).sort_values("売上", ascending=False)

by_pref.head(10)

In [ ]:
# 商品別
by_product = df.groupby(["product_code", "product_name", "category"]).agg(
    売上=("amount", "sum"),
    利益=("profit", "sum"),
    数量=("quantity", "sum"),
).sort_values("売上", ascending=False)

by_product

`groupby(["列A", "列B"])` のように **リストで複数列を渡せる** のもポイント。Excel ピボットの「行ラベルに複数フィールド」と同じ。

## 5. `pivot_table` — 地域 × カテゴリ のクロス集計

**「縦に地域、横にカテゴリ、中身は売上合計」** という Excel ピボット定番の形は、**`pivot_table`** が最短です。3-3 で使ったのと同じ関数です。

In [ ]:
pivot_region_cat = df.pivot_table(
    index="region",         # 行: 地域
    columns="category",     # 列: カテゴリ (定番果物 / 高級果物)
    values="amount",        # 値: 売上
    aggfunc="sum",          # 集計方法: 合計
    fill_value=0,           # 値がないセルは 0 で埋める
)
pivot_region_cat

In [ ]:
# 行合計・列合計を追加 (Excel ピボットの「総計」と同じ)
pivot_region_cat.assign(合計=pivot_region_cat.sum(axis=1))

> 💡 **`margins=True`** を `pivot_table` に渡せば、行・列の総計を自動で付けることもできます: `df.pivot_table(..., margins=True, margins_name="合計")`

## 6. 日次集計 (4-4 のグラフ用にもう一仕込み)

次の 4-4 のグラフ用に、**日次の売上推移** も作っておきましょう。

In [ ]:
by_date = df.groupby("date")["amount"].sum()
print(f"営業日数: {len(by_date)} 日")
by_date.head()

ここで作った **`by_region` / `by_pref` / `by_product` / `pivot_region_cat` / `by_date`** が、4-4 でそのままグラフ化される素材になります。

## 練習問題

本文のコードを少しだけ変える形の演習です。`df` をそのまま使ってください。

1. **カテゴリ別の売上・利益** を出してください。本文 cell-14 の **`region` を `category` に変える** だけ。
2. **県別 利益 TOP5** を出してください。本文 cell-16 の `.sort_values("売上", ...)` を **`"利益"` に変える** だけ。
3. **`pivot_table` で「行: 地域、列: 商品名、値: 売上合計」** を作ってください。本文 cell-20 の **`columns="category"` を `"product_name"` に変える** だけ。

In [ ]:
# ここに書いてみてください


<details>
<summary>答えを見る</summary>

```python
# 1. カテゴリ別 売上・利益
df.groupby("category").agg(
    売上=("amount", "sum"),
    利益=("profit", "sum"),
)

# 2. 県別 利益 TOP5
df.groupby("prefecture_name").agg(
    売上=("amount", "sum"),
    利益=("profit", "sum"),
).sort_values("利益", ascending=False).head(5)

# 3. 地域 × 商品名 売上のクロス集計
df.pivot_table(
    index="region",
    columns="product_name",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
```

</details>

## よくあるエラー

### 1. `merge` 後に行数が増えている
→ マスタ側のキーが **重複** している。1 商品コードに 2 行ある等。`master.duplicated(subset="product_code").sum()` で重複確認。マスタは **キーがユニーク** であるべき。

### 2. `merge` 後にカテゴリが `NaN` になる
→ マスタに無い商品コードがある。`df[df["category"].isna()]["product_code"].unique()` で漏れている商品コードを確認。

### 3. `KeyError: 'prefecture_code'` で merge 失敗
→ 左右の DataFrame で列名が違う。4-2 で `prefecture_code` 列を作ったのと、マスタの列名 `prefecture_code` が一致している必要がある。違う名前の時は `left_on=` `right_on=` で指定。

### 4. `groupby` した結果がインデックスになっていて使いづらい
→ `.reset_index()` を付けると、グループ列が通常の列に戻る。Excel に書き出すときに便利。

### 5. `pivot_table` の値が想定と違う
→ `aggfunc=` の指定漏れ (デフォルトは平均)。**合計したいときは `aggfunc="sum"`** を明示する。

## まとめ

- **`merge(master, on="キー", how="left")`** で VLOOKUP 相当の処理が 1 行で
- 必要な列だけ持ってくるには `master[["キー", "欲しい列"]]` で絞ってから merge
- **`groupby(...).agg(列名=("元列", "集計"))`** で複数の指標を 1 つの表に
- **`pivot_table(index=, columns=, values=, aggfunc=)`** で Excel ピボット相当のクロス集計
- 今回作った `by_region` / `by_pref` / `by_product` / `pivot_region_cat` / `by_date` を 4-4 でグラフにし、4-5 で Excel に書き出す

47,000 行を扱っているのに、ここまで全部 **2-3 / 2-4 で習った技** だけで進んでいます。3 章までの内容が部品としてそのまま効いていることが分かるはずです。

次の **4-4** では、今作った集計結果を **3 章の matplotlib スタイル** で図にして、PNG として保存します。